### Churn Prediction with KNN
Author: Marc Tanguy
Description: Predict whether a type of costumer is going to churn using KNN

#### Step 1: Load and explore the data

In [1]:
import pandas as pd

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(df.shape)
print(df.head())
print(df.dtypes)

df['Churn'].value_counts(normalize=True)


(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64

In [2]:
# Target balance
print("=== Churn Balance ===")
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(2))

# Contract type
print("\n=== Contract Type ===")
print(df['Contract'].value_counts())

# Tenure distribution (numeric, so we bin it)
print("\n=== Tenure Groups ===")
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72], labels=['0-12m', '13-24m', '25-48m', '49-72m'])
print(df['tenure_group'].value_counts().sort_index())

# Monthly Charges
print("\n=== Monthly Charges ===")
print(df['MonthlyCharges'].describe().round(2))

# Total Charges - note that this column is object type, so we need to convert it to numeric first
print("\n=== Total Charges ===")
print(df['TotalCharges'].apply(pd.to_numeric, errors='coerce').describe().round(2))

=== Churn Balance ===
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.73
Yes    0.27
Name: proportion, dtype: float64

=== Contract Type ===
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

=== Tenure Groups ===
tenure_group
0-12m     2175
13-24m    1024
25-48m    1594
49-72m    2239
Name: count, dtype: int64

=== Monthly Charges ===
count    7043.00
mean       64.76
std        30.09
min        18.25
25%        35.50
50%        70.35
75%        89.85
max       118.75
Name: MonthlyCharges, dtype: float64

=== Total Charges ===
count    7032.00
mean     2283.30
std      2266.77
min        18.80
25%       401.45
50%      1397.48
75%      3794.74
max      8684.80
Name: TotalCharges, dtype: float64


#### Step 2: Data preprocessing

In [3]:
# Missing values per column
print(df.isnull().sum())
print(df.isnull().mean().round(2))

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
tenure_group        11
dtype: int64
customerID          0.0
gender              0.0
SeniorCitizen       0.0
Partner             0.0
Dependents          0.0
tenure              0.0
PhoneService        0.0
MultipleLines       0.0
InternetService     0.0
OnlineSecurity      0.0
OnlineBackup        0.0
DeviceProtection    0.0
TechSupport         0.0
StreamingTV         0.0
StreamingMovies     0.0
Contract            0.0
PaperlessBilling    0.0
PaymentMethod       0.0
MonthlyCharges      0.0
TotalCharges        0.0
C

In [ ]:
# 1. Fix TotalCharges (coerce blanks to NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill TotalCharges NaNs with median (only 11 rows affected)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# 2. Drop customerID (not useful for modeling) - other columns are all relevant for modeling, so we keep them
df.drop(columns=['customerID'], inplace=True)

# Drop the tenure_group we created earlier (it was just for EDA)
if 'tenure_group' in df.columns:
    df.drop(columns=['tenure_group'], inplace=True)

# 3. Encode target variable / convert target to binary
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 4. Binary columns — map Yes/No to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

# Gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

# One-hot encode multi-category columns
cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 5. Separate features (X) from target (y)
X = df.drop(columns=['Churn'])
y = df['Churn']

print(X.shape)
print(y.value_counts())


(7043, 30)
Churn
0    5174
1    1869
Name: count, dtype: int64


#### Step 3: Split the data

In [5]:
from sklearn.model_selection import train_test_split

# Split into train and test sets (80% train, 20% test), stratified to maintain churn ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("Train set churn distribution:")
print(y_train.value_counts(normalize=True).round(2))
print("Test set churn distribution:")
print(y_test.value_counts(normalize=True).round(2))

Train set shape: (5634, 30)
Test set shape: (1409, 30)
Train set churn distribution:
Churn
0    0.73
1    0.27
Name: proportion, dtype: float64
Test set churn distribution:
Churn
0    0.73
1    0.27
Name: proportion, dtype: float64


#### Step 4: Train a KKK Model